In [ ]:
import opendatasets as od
od.download('https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: abdalrahman70
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia


100%|██████████| 2.29G/2.29G [00:23<00:00, 105MB/s]


In [ ]:
import os
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.metrics import accuracy_score

In [ ]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')
device

'cuda'

In [ ]:
# Create Dataset Reader
class DataReader(Dataset):

    def __init__(self, root_dir, transformer=None):
        self.root_dir = root_dir
        self.transformer = transformer
        self.images_paths = []
        self.labels = []

        for label in ['NORMAL', 'PNEUMONIA']:
            class_dir = os.path.join(root_dir, label)
            for image in os.listdir(class_dir):
                self.images_paths.append(os.path.join(class_dir, image))
                self.labels.append(0 if label == 'NORMAL' else 1)

    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, idx):
        image_path = self.images_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert('RGB')

        if self.transformer:
            image = self.transformer(image)

        return image, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.486, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Read Data
train_data = DataReader(root_dir='/content/chest-xray-pneumonia/chest_xray/train', transformer=transform)
test_data = DataReader(root_dir='/content/chest-xray-pneumonia/chest_xray/test', transformer=transform)
val_data = DataReader(root_dir='/content/chest-xray-pneumonia/chest_xray/val', transformer=transform)

In [ ]:
# Load Data
train_loader = DataLoader(dataset=train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=32, shuffle=False)
val_loader = DataLoader(dataset=val_data, batch_size=32, shuffle=False)

In [ ]:
# Create ResNet18 Model
model = models.resnet18(weights= models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Train the model
for epoch in range(10):
    model.train()
    running_loss = 0
    train_labels = []
    train_preds = []

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        train_labels.extend(labels.cpu().numpy())
        train_preds.extend(preds.cpu().numpy())
        running_loss += loss

    print(f'Epoch: {epoch+1}, Loss: {running_loss/ len(train_loader)}')
    print('Train Accuracy: ',accuracy_score(train_labels, train_preds))

    model.eval()
    val_labels = []
    val_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            output = model(images)
            _, preds = torch.max(output, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())

        print('Validation Accuracy: ', accuracy_score(val_labels, val_preds))

Epoch: 1, Loss: 0.10651213675737381
Train Accuracy:  0.95954754601227
Validation Accuracy:  0.9375
Epoch: 2, Loss: 0.059999797493219376
Train Accuracy:  0.9766104294478528
Validation Accuracy:  0.75
Epoch: 3, Loss: 0.04674386605620384
Train Accuracy:  0.9825536809815951
Validation Accuracy:  0.875
Epoch: 4, Loss: 0.026518957689404488
Train Accuracy:  0.991180981595092
Validation Accuracy:  0.875
Epoch: 5, Loss: 0.030733494088053703
Train Accuracy:  0.9884969325153374
Validation Accuracy:  1.0
Epoch: 6, Loss: 0.034464359283447266
Train Accuracy:  0.9886886503067485
Validation Accuracy:  1.0
Epoch: 7, Loss: 0.048629652708768845
Train Accuracy:  0.9819785276073619
Validation Accuracy:  1.0
Epoch: 8, Loss: 0.022694619372487068
Train Accuracy:  0.9927147239263804
Validation Accuracy:  0.875
Epoch: 9, Loss: 0.010142363607883453
Train Accuracy:  0.9965490797546013
Validation Accuracy:  0.8125
Epoch: 10, Loss: 0.012151953764259815
Train Accuracy:  0.9963573619631901
Validation Accuracy:  0.937

In [ ]:
# Test the model
model.eval()
test_labels = []
test_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        output = model(images)
        _, pred = torch.max(output, 1)
        test_labels.extend(labels.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())

    print('Test Accuracy: ', accuracy_score(val_labels, val_preds))

Test Accuracy:  0.9375


In [ ]:
torch.save(model.state_dict(), 'model_Xray.pth')